# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load and explore the FAIR^2 dataset—*Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution*—using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema and accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We will load the dataset metadata and records via the Croissant schema using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")

## 2. Data Overview
Let's review the available record sets and their fields and identify their `@id` values for reference in subsequent steps.

In [ ]:
# List all record sets in the dataset using their `@id`

record_sets = dataset.record_sets
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"- @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Number of fields: {len(rs.fields)}")
    print(f"    Field @ids:")
    for field in rs.fields:
        print(f"      - {field.id}")
    print()

## 3. Data Extraction
Now we load the data from each record set into Pandas DataFrames for further exploration. All data entity references are by their `@id` values.

In [ ]:
# Use all available record sets
record_set_ids = [rs.id for rs in dataset.record_sets]
dfs = {}

for rs_id in record_set_ids:
    # Load records for this record set by its `@id`
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id}.")
    print(f"Columns (@id): {list(df.columns)}\n")

# Preview the first record set if available
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"Preview of the first record set '@id': {preview_id}")
    display(dfs[preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the dataset further. We'll select a numeric field (by its `@id`), filter records, normalize the field, and perform group-based aggregations.

> **Note:** Replace the field `@id` and group field `@id` with actual values printed above for your use case.

In [ ]:
# Choose a record set and numeric field @ids
# Example only: replace with actual @ids as appropriate
record_set_id = record_set_ids[0] if record_set_ids else None

# Inspect columns in the chosen DataFrame
if record_set_id:
    print("Columns available:", dfs[record_set_id].columns.tolist())

# You should set these to meaningful field @ids for your dataset:
# For illustration, let's try to auto-detect a numeric field
if record_set_id:
    df = dfs[record_set_id]
    # Select the first numeric-looking column (heuristic)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Using numeric field (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() > 0 else 0
        # Filter by mean as threshold (illustrative)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} (z-score):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field @id
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by field (@id): {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped.head())
        else:
            print('No categorical field found for grouping.')
    else:
        print('No numeric field found for filtering/normalization.')

## 5. Visualization
Visualize distributions or relationships in the dataset (e.g., histograms, box plots, or group-based bar plots).

In [ ]:
# Visualization example: Histogram and group-wise average (if data available)
import matplotlib.pyplot as plt

if record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    dfs[record_set_id][numeric_field_id].hist(bins=15)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Group-wise bar plot
    if group_field_id:
        plt.figure(figsize=(8,4))
        grouped_df = dfs[record_set_id].groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        grouped_df.plot(kind='bar')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and visualized data from the FAIR² "Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer" dataset using the `mlcroissant` library. We referenced all data entities by their `@id` as per the Croissant schema, loaded tabular record sets into DataFrames, and applied basic EDA and visualizations. For deeper analysis and modeling, refine your field selection and processing steps as needed.